# Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual <a href='https://quantra.quantinsti.com/quantra-notebook' target="_blank">here</a>.
1. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
1. Before running this notebook on your local PC:<br>
i.  You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on "**Run Codes Locally on Your Machine**" in the course.<br>
ii. You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or python modules that might be required to run this notebook.

# Monitoring & Managing Your Orders

In the last notebook, we learned how to place orders. But what happens after you've sent an order to the exchange? How do you track its status, change it if the market moves, or cancel it if your strategy changes? 

In this notebook, we'll cover the essential tools for post-placement order management. You will learn how to:
-   View all your open and executed orders.
-   Modify the parameters of a pending order.
-   Cancel a pending order.
-   Check the detailed history of a specific order and your daily trade book.
<BR><BR>

<font color='red'>
<b>Extreme Caution</b>: The code in this notebook places <b>REAL</b> orders in the market. Executing these cells during market hours with a funded account might result in actual trades. 
Proceed with extreme caution and at your own risk. It is highly recommended to test with a small quantity or use a test account if possible.
</font>

### 1. Authentication

As always, we begin by authenticating our Kite Connect client.

In [ ]:
from kiteconnect import KiteConnect
import logging
import pandas as pd # We'll use pandas to display data cleanly

logging.basicConfig(level=logging.INFO)

# --- Credentials ---
import os
API_KEY = os.environ.get('ZERODHA_ALGOTRADER_API_KEY')
API_SECRET = os.environ.get('ZERODHA_ALGOTRADER_API_SECRET')

access_token = None
kite = None

# --- Authentication ---
if api_key != "YOUR_API_KEY":
    try:
        kite = KiteConnect(api_key=api_key)
        print("Login URL:", kite.login_url())
        request_token = input("Enter request_token: ").strip()
        
        # Generate session and get access token
        data = kite.generate_session(request_token, api_secret=api_secret)
        access_token = data['access_token']
        
        # Set access token
        kite.set_access_token(access_token)
        
        # Verify authentication
        profile = kite.profile()
        logging.info(f"Logged in as: {profile['user_name']}")
        logging.info(f"Access Token: {access_token}")
        
    except Exception as e:
        logging.error(f"Authentication failed: {e}")
else:
    print("Please update your API_KEY and API_SECRET to run.")

### 2. Setting Up a Sample Order

To learn how to manage an order, we first need an open order to manage! Let's place a LIMIT buy order for INFY at a price that is significantly lower than the current market price. This ensures the order will remain `OPEN` and won't execute immediately, allowing us to modify and cancel it in the next steps.

In [ ]:
try:
    # Fetch the last traded price to set a realistic but unfilled limit price
    ltp_data = kite.ltp('NSE:INFY')
    last_price = ltp_data['NSE:INFY']['last_price']
    limit_price = round(last_price * 0.95, 1) # Set a limit price 5% below the LTP
    logging.info(f"Current INFY LTP is {last_price}. Placing LIMIT BUY order at {limit_price}")

    order_id_to_manage = kite.place_order(
        variety=kite.VARIETY_REGULAR,
        exchange=kite.EXCHANGE_NSE,
        tradingsymbol='INFY',
        transaction_type=kite.TRANSACTION_TYPE_BUY,
        quantity=1,
        product=kite.PRODUCT_CNC,
        order_type=kite.ORDER_TYPE_LIMIT,
        price=limit_price
    )
    logging.info(f"Sample LIMIT order placed successfully. Order ID: {order_id_to_manage}")
except Exception as e:
    logging.error(f"Error placing sample order: {e}")

### 3. Viewing All Orders

The `kite.orders()` method fetches a list of all orders placed during the day. The raw output is very detailed, so we'll use the pandas library to format it into a clean, readable table.

In [ ]:
try:
    all_orders = kite.orders()
    # Convert the list of order dictionaries into a pandas DataFrame
    orders_df = pd.DataFrame(all_orders)
    
    # Display only the most relevant columns
    if not orders_df.empty:
        display(orders_df[['order_timestamp', 'tradingsymbol', 'transaction_type', 'quantity', 'price', 'status', 'order_id']])
    else:
        logging.info("No orders found for today.")
        
except Exception as e:
    logging.error(f"Error fetching orders: {e}")

To learn more about what a particular order status indicates, you can visit this link: [Kite Order Status](https://kite.trade/docs/connect/v3/orders/#order-statuses)

### 4. Modifying an Open Order

Our limit order for INFY is still `OPEN`, but the market has moved up, and it's unlikely to be filled. We can use the `kite.modify_order()` method to change its price to be closer to the current market price.

This method requires the `order_id` of the specific order you wish to change.

In [ ]:
try:
    # Let's modify the price of our pending order
    new_limit_price = round(last_price * 0.96, 1) # New price (lower to 4%)
    
    kite.modify_order(
        variety=kite.VARIETY_REGULAR,
        order_id=order_id_to_manage,
        price=new_limit_price
    )
    logging.info(f"Successfully modified order {order_id_to_manage} to new price: {new_limit_price}")
except Exception as e:
    logging.error(f"Error modifying order: {e}")

### 5. Cancelling an Open Order

What if our view on the stock changes completely? We no longer want to buy INFY, so we must cancel our pending order to prevent it from executing.

We use the `kite.cancel_order()` method, which also requires the specific `order_id`.

In [ ]:
try:
    kite.cancel_order(
        variety=kite.VARIETY_REGULAR,
        order_id=order_id_to_manage
    )
    logging.info(f"Successfully placed cancel request for order {order_id_to_manage}.")
except Exception as e:
    logging.error(f"Error cancelling order: {e}")

### 6. Viewing History and Trades

The API also provides functions to see the detailed history of a single order and a list of all your executed trades (your trade book).

-   `kite.order_history(order_id)`: Fetches the complete lifecycle of a specific order, including status changes like 'CANCELLED' or 'REJECTED'.
-   `kite.trades()`: Fetches a list of only the orders that were actually filled.

In [ ]:
try:
    # Get the history of the order we just managed
    history = kite.order_history(order_id=order_id_to_manage)
    logging.info(f"History for order {order_id_to_manage}:")
    display(pd.DataFrame(history))

    # Get the tradebook for the day
    trades = kite.trades()
    logging.info("Daily Tradebook:")
    if trades:
        display(pd.DataFrame(trades))
    else:
        logging.info("No trades executed today.")
except Exception as e:
    logging.error(f"Error fetching history or trades: {e}")

### 7. Conclusion & Next Steps

Congratulations! You now have a complete set of tools to manage your orders. You can view all orders, modify pending ones, cancel them, and review your history and filled trades.

This is an important skill set for building any robust automated trading system, as it allows your code to react to changing market conditions and manage its own lifecycle. 

Note that here you used the `order_id` to modify or get history of an order. You can also use postback method, which involves setting up a public url so that you can get updates of your orders reliably, irrespective of when they happen. For more information, you can check the documentation at the following link: <a href="https://kite.trade/docs/connect/v3/postbacks/" target="_blank" rel="noopener">Postback (WebHooks)</a>


In the next video, we'll dive into managing positions.<br><br>